# Распознавание речи (SpeechToText - STT)

## Бесплатное использование STT: SpeechRecognition

In [ ]:
!pip -q install ffmpeg-python SpeechRecognition pydub jiwer

# Загружаем модуль для работы с микрофоном
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Intensive/v2.0/micro.py', None, quiet=True)

import micro                        # модуль для работы с микрофоном
import scipy
import os
import speech_recognition as sR     # модуль для распознавания речи
from jiwer import wer               # модуль для оценки распознанного текста по методу WER
from pydub import AudioSegment      # модуль конвертирования аудио
from IPython.display import Audio
from pprint import pprint as pp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 59.2 MB/s eta 0:00:00


In [ ]:
# Функция для распознавания аудио
def recognizeAudio(filename, duration=None):
    r = sR.Recognizer() # создаем объект класса Recognizer
    with sR.AudioFile(filename) as source:
        audio = r.record(source, duration=duration) # считываем аудиофайл
    return r.recognize_google(audio, language='ru') # запускаем распознавание


# Конвертация аудиофайла из формата MP3 в WAV
sound = AudioSegment.from_mp3('audio.mp3')
sound.export('audio.wav', format="wav")
Audio('audio.wav')

**Конвертация аудиофайлов может быть полезна по нескольким причинам:**

- **Совместимость**: Некоторые приложения или библиотеки могут поддерживать только определенные форматы аудио. WAV — это более универсальный формат, который часто используется в обработке звука.
  
- **Качество**: WAV — это формат без сжатия, который сохраняет аудиоданные в их оригинальном качестве. Это может быть важно, если вы планируете проводить дальнейшую обработку звука.

- **Удобство**: Иногда может потребоваться конвертировать файлы для удобства работы с ними в других инструментах или библиотеках.


In [ ]:
# Запускаем распознавание
res = recognizeAudio('audio.wav')
pp(res)

('Ночь Улица Фонарь Аптека Бессмысленный и тусклый свет Живи ещё хоть четверть '
 'века всё будет так Исхода нет умрёшь начнёшь опять сначала и повторится всё '
 'как встарь ночь ледяная рябь канала аптека улица фонарь')


In [ ]:
# Сравнение оригинального текста и результата распознавания

original = "Ночь улица фонарь аптека Бессмысленный и тусклый свет Живи ещё хоть четверть века \
Всё будет так Исхода нет Умрёшь начнёшь опять сначала И повторится всё как встарь \
Ночь ледяная рябь канала Аптека улица фонарь"

print('Оригинал:\n')
pp(original)
print('\nРезультат распознавания:\n')
pp(res)

# Оценка качества распознавания: Вывод оригинального текста, результата распознавания и метрики WER
# позволяет оценить, насколько хорошо система распознавания справилась с задачей. Это полезно для
# анализа и улучшения качества распознавания речи.
print('\nWER:', wer(original.lower(), res.lower())) # считаем метрику качества

Оригинал:

('Ночь улица фонарь аптека Бессмысленный и тусклый свет Живи ещё хоть четверть '
 'века Всё будет так Исхода нет Умрёшь начнёшь опять сначала И повторится всё '
 'как встарь Ночь ледяная рябь канала Аптека улица фонарь')

Результат распознавания:

('Ночь Улица Фонарь Аптека Бессмысленный и тусклый свет Живи ещё хоть четверть '
 'века всё будет так Исхода нет умрёшь начнёшь опять сначала и повторится всё '
 'как встарь ночь ледяная рябь канала аптека улица фонарь')

WER: 0.0


## STT: OpenAI Whisper

In [ ]:
!pip install -q openai==1.59.9

import openai
from google.colab import userdata

# Получение ключа OpenAI API из секретов Колаба
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.5/455.5 kB 7.2 MB/s eta 0:00:00


In [ ]:
def stt_whisper_online(voice_file, model='whisper-1'):
    with open(voice_file, "rb") as audio_file:
        transcript = openai.audio.transcriptions.create(
            model=model, file=audio_file)
    return transcript.text


pp(stt_whisper_online('audio.mp3'))

('Ночь. Улица. Фонарь. Аптека. Бессмысленный и тусклый свет. Живи ещё. Хоть '
 'четверть века. Всё будет так. Исхода нет. Умрёшь, начнёшь опять сначала, и '
 'повторится всё, как встарь. Ночь. Ледяная рябь. Канала. Аптека. Улица. ')


## Записывает аудио с микрофона и сохраняет его в файл

In [ ]:
# Записывает аудио с микрофона и сохраняет его в файл
def record():
    # Запустим процесс записи с микрофона
    audio, sr = micro.get_audio()
    # Сохраним запись в файл recording.wav
    scipy.io.wavfile.write('recording.wav', sr, audio)


# Запись с микрофона
record()

In [ ]:
original = "мороз и солнце день чудесный ещё ты дремлешь друг прелестный пора красавица проснись \
Открой сомкнуты негой взоры Навстречу северной Авроры звездою севера явись"

res = recognizeAudio('recording.wav')

print('Оригинал:\n')
pp(original)
print('\nРезультат распознавания:\n')
pp(res)

print('\nWER:', wer(original.lower(), res.lower()))

Оригинал:

('мороз и солнце день чудесный ещё ты дремлешь друг прелестный пора красавица '
 'проснись Открой сомкнуты негой взоры Навстречу северной Авроры звездою '
 'севера явись')

Результат распознавания:

('Мороз и солнце день чудесный Ещё ты дремлешь друг прелестный Пора красавица '
 'проснись открой сомкнуты негой взоры навстречу северной Авроры звездою '
 'севера Я весь')

WER: 0.08695652173913043


## vosk-model-small-ru-0.22

In [ ]:
!pip install vosk soundfile
!apt-get install ffmpeg -qq

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 56.9 MB/s eta 0:00:00
  Created wheel for srt: filename=srt-3.5.3-py3-none-any.whl size=22427 sha256=3385692e0970e2f3103fdec15e04e945a53b3d92b83eeb7d0db830407433d139
  Stored in directory: /root/.cache/pip/wheels/1f/43/f1/23ee9119497fcb57d9f7046fbf34c6d9027c46a1fa7824cf08
Successfully built srt


In [ ]:
import os
import wave
import json
from vosk import Model, KaldiRecognizer
from IPython.display import Audio, display

In [ ]:
# Задаём путь к исходному MP3-файлу
mp3_file = "/content/audio.mp3"

# Задаём путь для временного WAV-файла
wav_file = "/content/temp.wav"

# Конвертация MP3 в WAV (16 кГц, моно, 16-bit) с помощью ffmpeg
!ffmpeg -i "{mp3_file}" -ar 16000 -ac 1 -sample_fmt s16 -y "{wav_file}"

# Отобразим аудио-плеер для прослушивания исходного MP3 (или сконвертированного WAV, если нужно)
display(Audio(mp3_file))

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [ ]:
# Проверяем наличие модели русского языка в папке "model-ru"
if not os.path.exists("model-ru"):
    print("Модель не найдена. Скачиваем и распаковываем модель для русского языка...")
    !wget -q https://alphacephei.com/vosk/models/vosk-model-small-ru-0.22.zip -O vosk-model-small-ru-0.22.zip
    !unzip -q vosk-model-small-ru-0.22.zip -d .
    if os.path.exists("vosk-model-small-ru-0.22"):
        os.rename("vosk-model-small-ru-0.22", "model-ru")
    print("Модель успешно загружена.")
else:
    print("Модель уже загружена.")

Модель не найдена. Скачиваем и распаковываем модель для русского языка...
Модель успешно загружена.


In [ ]:
# Открываем сконвертированный WAV-файл
try:
    wf = wave.open(wav_file, "rb")
except Exception as e:
    print("Ошибка при открытии аудиофайла:", e)
    raise

# Проверяем, что аудиофайл соответствует требованиям: WAV, 16 кГц, моно, 16-bit
if wf.getnchannels() != 1 or wf.getsampwidth() != 2 or wf.getframerate() != 16000:
    print("Аудиофайл должен быть в формате WAV, 16 кГц, моно, 16-bit. Пожалуйста, конвертируйте файл.")
else:
    # Инициализация модели
    try:
        model = Model("model-ru")
    except Exception as e:
        print("Ошибка при загрузке модели:", e)
        raise

    recognizer = KaldiRecognizer(model, wf.getframerate())

    result_text = ""
    tokens = []

    # Читаем аудио порциями и распознаем речь
    while True:
        data = wf.readframes(4000)
        if len(data) == 0:
            break
        if recognizer.AcceptWaveform(data):
            result_json = recognizer.Result()
            parsed_result = json.loads(result_json)
            result_text += parsed_result.get("text", "") + " "
            if "result" in parsed_result:
                tokens.extend(parsed_result["result"])

    # Обрабатываем оставшийся результат
    final_result = recognizer.FinalResult()
    parsed_final = json.loads(final_result)
    result_text += parsed_final.get("text", "")
    if "result" in parsed_final:
        tokens.extend(parsed_final["result"])

    print("Распознанный текст:")
    print(result_text.strip())

    # Если токены отсутствуют, создаём приближённые тайм-коды
    if not tokens:
        print("\nТокены с тайм-кодами не обнаружены. Создаём приблизительные тайм-коды:")
        # Общая длительность аудио в секундах
        total_frames = wf.getnframes()
        total_duration = total_frames / wf.getframerate()
        # Разбиваем транскрипт на слова
        words = result_text.strip().split()
        num_words = len(words)
        if num_words == 0:
            print("Нет слов для создания тайм-кодов.")
        else:
            time_per_word = total_duration / num_words
            fallback_tokens = []
            current_time = 0.0
            for word in words:
                token = {"word": word, "start": current_time, "end": current_time + time_per_word}
                fallback_tokens.append(token)
                current_time += time_per_word
            tokens = fallback_tokens

    print("\nТайм-коды для каждого слова:")
    for token in tokens:
        word = token.get("word", "")
        start = token.get("start", 0)
        end = token.get("end", 0)
        print(f"{word}: start={start:.2f}, end={end:.2f}")

Распознанный текст:
ночь улица фонарь аптека бессмысленные и тусклый свет живи ещё хоть четверть века все будет так исхода нет умрёшь начнёшь опять сначала и повторится все как встарь ночь ледяная рябь канала аптека улица фонарь

Токены с тайм-кодами не обнаружены. Создаём приблизительные тайм-коды:

Тайм-коды для каждого слова:
ночь: start=0.00, end=0.70
улица: start=0.70, end=1.40
фонарь: start=1.40, end=2.09
аптека: start=2.09, end=2.79
бессмысленные: start=2.79, end=3.49
и: start=3.49, end=4.19
тусклый: start=4.19, end=4.88
свет: start=4.88, end=5.58
живи: start=5.58, end=6.28
ещё: start=6.28, end=6.98
хоть: start=6.98, end=7.67
четверть: start=7.67, end=8.37
века: start=8.37, end=9.07
все: start=9.07, end=9.77
будет: start=9.77, end=10.46
так: start=10.46, end=11.16
исхода: start=11.16, end=11.86
нет: start=11.86, end=12.56
умрёшь: start=12.56, end=13.25
начнёшь: start=13.25, end=13.95
опять: start=13.95, end=14.65
сначала: start=14.65, end=15.35
и: start=15.35, end=16.05
повторит

## STT: google-cloud-speech

In [ ]:
!pip install -q google-cloud-speech requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.1/334.1 kB 4.5 MB/s eta 0:00:00


In [ ]:
import requests
import base64
import json
from google.cloud import speech
from google.colab import userdata

In [ ]:
# Нужна регистрация на https://cloud.google.com/
# Получаем API-ключ из защищённого хранилища Colab
os.environ["GOOGLE_CLOUD_API_KEY"] = userdata.get("GOOGLE_CLOUD_API_KEY")

In [ ]:
# Определяем функцию для отправки аудиофайла в API Google Speech-to-Text
def stt_google(voice_file_path, language_code="ru-RU", sample_rate_hertz=48000, encoding="MP3"):
    # Считываем аудиофайл и кодируем его в формат base64
    with open(voice_file_path, "rb") as f:
        audio_content = base64.b64encode(f.read()).decode("utf-8")

    # Формируем тело запроса
    # url = f"https://speech.googleapis.com/v1/speech:recognize?key={GOOGLE_CLOUD_API_KEY}"
    url = f"https://speech.googleapis.com/v1/speech:recognize?key={os.environ['GOOGLE_CLOUD_API_KEY']}"

    headers = {
        "Content-Type": "application/json; charset=utf-8"
    }

    body = {
        "config": {
            "encoding": encoding,
            "sampleRateHertz": sample_rate_hertz,
            "languageCode": language_code
        },
        "audio": {
            "content": audio_content
        }
    }

    # Отправляем POST-запрос в API
    response = requests.post(url, headers=headers, data=json.dumps(body))

    # Обрабатываем ответ
    if response.status_code != 200:
        print("Ошибка запроса:", response.status_code, response.text)
        return None

    response_json = response.json()
    if "results" in response_json:
        # Возвращаем транскрипцию первого результата
        return response_json["results"][0]["alternatives"][0]["transcript"]
    else:
        print("Результаты не найдены.")
        return None


# Запускаем функцию и выводим результат транскрипции
transcripcion = stt_google("/content/audio.mp3", language_code="ru-RU")
print("Транскрипция:", transcripcion)


Транскрипция: Ночь Улица Фонарь Аптека Бессмысленный и тусклый свет Живи ещё хоть четверть века всё будет так Исхода нет умрёшь начнёшь опять сначала и повторится всё как встарь ночь ледяная рябь канала аптека улица фонарь


In [ ]:
pp(transcripcion)

('Ночь Улица Фонарь Аптека Бессмысленный и тусклый свет Живи ещё хоть четверть '
 'века всё будет так Исхода нет умрёшь начнёшь опять сначала и повторится всё '
 'как встарь ночь ледяная рябь канала аптека улица фонарь')


Дополнительные материалы для индивидуального изучения: [Инструмменты SpeechToText](https://colab.research.google.com/drive/1osXMq_H81dSpwJkM-aimdhDFRF6PSaPN?usp=sharing)